# Phase 6: Location & Area Analysis
## Geographic Analysis of Restaurant Distribution
**Objective:** Analyze restaurant distribution, area performance, and geographic insights
**Output:** Location-based dashboard insights for market analysis

In [4]:
# Import Required Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

print("✓ Libraries imported")

✓ Libraries imported


In [5]:
# Load cleaned data
df = pd.read_csv('Zomato_datasets/zomato_cleaned.csv')

print(f"Dataset loaded: {df.shape}")
print(f"Total unique areas: {df['area'].nunique()}")
print(f"\nTop 10 Areas:")
print(df['area'].value_counts().head(10))

Dataset loaded: (7105, 12)
Total unique areas: 30

Top 10 Areas:
area
Byresandra,Tavarekere,Madiwala    798
Bannerghatta Road                 552
Brookefield                       477
Brigade Road                      464
Indiranagar                       455
Electronic City                   403
Malleshwaram                      402
Kalyan Nagar                      384
Bellandur                         361
Banashankari                      359
Name: count, dtype: int64


In [6]:
# 1. Top 20 Areas by Restaurant Count
top_areas = df['area'].value_counts().head(20)

fig = px.barh(
    x=top_areas.values,
    y=top_areas.index,
    title='Top 20 Areas by Number of Restaurants',
    labels={'x': 'Number of Restaurants', 'y': 'Area'},
    color=top_areas.values,
    color_continuous_scale='Blues'
)

fig.update_layout(height=800, showlegend=False)
fig.show()

print(f"\nTop 20 Areas by Restaurant Count:")
print(top_areas)

AttributeError: module 'plotly.express' has no attribute 'barh'

In [ ]:
# 2. Area Performance Analysis
area_performance = df.groupby('area').agg({
    'restaurant_name': 'count',
    'rating': ['mean', 'std'],
    'avg_cost': 'mean',
    'num_ratings': 'sum',
    'has_online_order': 'mean',
    'has_table_booking': 'mean'
}).reset_index()

area_performance.columns = ['area', 'restaurant_count', 'avg_rating', 'rating_std', 'avg_cost', 'total_ratings', 'online_order_pct', 'table_booking_pct']
area_performance = area_performance.sort_values('restaurant_count', ascending=False).head(15)

# Visualize top areas performance
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Average Rating', 'Average Cost', 'Online Order Availability', 'Table Booking Availability')
)

fig.add_trace(
    go.Bar(x=area_performance['area'], y=area_performance['avg_rating'], marker_color='lightblue', name='Rating'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=area_performance['area'], y=area_performance['avg_cost'], marker_color='lightgreen', name='Cost'),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=area_performance['area'], y=area_performance['online_order_pct'], marker_color='lightyellow', name='Online %'),
    row=2, col=1
)

fig.add_trace(
    go.Bar(x=area_performance['area'], y=area_performance['table_booking_pct'], marker_color='lightcoral', name='Booking %'),
    row=2, col=2
)

fig.update_xaxes(tickangle=-45)
fig.update_layout(height=900, title_text='Top 15 Areas - Performance Metrics', showlegend=False)
fig.show()

print("\nTop 15 Areas - Performance Summary:")
print(area_performance[['area', 'restaurant_count', 'avg_rating', 'avg_cost']].to_string())

In [ ]:
# 3. Area Concentration Analysis
area_counts = df['area'].value_counts()
concentration = area_counts.head(10).sum() / len(df) * 100

fig = go.Figure()

fig.add_trace(go.Pie(
    labels=['Top 10 Areas', 'Other Areas'],
    values=[area_counts.head(10).sum(), area_counts.iloc[10:].sum()],
    marker=dict(colors=['#3498db', '#95a5a6'])
))

fig.update_layout(
    title='Market Concentration - Top 10 Areas',
    height=500
)

fig.show()

print(f"\nMarket Concentration Analysis:")
print(f"  Top 10 Areas: {concentration:.1f}% of restaurants")
print(f"  Total unique areas: {df['area'].nunique()}")

In [ ]:
# 4. Restaurant Type Distribution by Area (Top 10 Areas)
top_10_areas = df['area'].value_counts().head(10).index
df_top = df[df['area'].isin(top_10_areas)]

area_type = df_top.groupby(['area', 'restaurant_type']).size().reset_index(name='count')
area_type_top = area_type.sort_values('count', ascending=False).groupby('area').head(5)

fig = px.bar(
    area_type_top,
    x='count',
    y='area',
    color='restaurant_type',
    title='Top Restaurant Types in Top 10 Areas',
    labels={'count': 'Count', 'area': 'Area', 'restaurant_type': 'Restaurant Type'},
    barmode='group'
)

fig.update_layout(height=700)
fig.show()

In [ ]:
# 5. Service Availability by Area (Top 15 Areas)
top_15_areas = df['area'].value_counts().head(15).index
service_by_area = df[df['area'].isin(top_15_areas)].groupby('area').agg({
    'has_online_order': lambda x: (x.sum() / len(x) * 100),
    'has_table_booking': lambda x: (x.sum() / len(x) * 100)
}).reset_index()

fig = go.Figure(data=[
    go.Bar(x=service_by_area['area'], y=service_by_area['has_online_order'], name='Online Order %', marker_color='green'),
    go.Bar(x=service_by_area['area'], y=service_by_area['has_table_booking'], name='Table Booking %', marker_color='blue')
])

fig.update_layout(
    title='Service Availability in Top 15 Areas',
    xaxis_tickangle=-45,
    height=600,
    barmode='group'
)

fig.show()

print("\nService Availability by Top 15 Areas:")
print(service_by_area.sort_values('has_online_order', ascending=False).to_string())

In [ ]:
# 6. Rating Distribution by Area (Top 10 Areas)
df_top_areas = df[df['area'].isin(df['area'].value_counts().head(10).index)]

fig = px.box(
    df_top_areas,
    x='area',
    y='rating',
    color='area',
    title='Rating Distribution in Top 10 Areas',
    labels={'area': 'Area', 'rating': 'Rating'},
    points='outliers'
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

In [ ]:
# 7. Price Distribution by Area (Top 10 Areas)
fig = px.violin(
    df_top_areas,
    x='area',
    y='avg_cost',
    color='area',
    title='Price Distribution in Top 10 Areas',
    labels={'area': 'Area', 'avg_cost': 'Average Cost (₹)'},
    points=False
)

fig.update_layout(height=600, xaxis_tickangle=-45, showlegend=False)
fig.show()

In [ ]:
# 8. Area Summary Statistics
print("\n" + "="*70)
print("LOCATION ANALYSIS SUMMARY")
print("="*70)

print(f"\nTotal Unique Areas: {df['area'].nunique()}")
print(f"\nTop 5 Areas by Count:")
print(df['area'].value_counts().head(5))

print(f"\n\nHigh-Performing Areas (Avg Rating >= 3.8, Min 10 restaurants):")
high_perf = df.groupby('area').filter(lambda x: len(x) >= 10).groupby('area').agg({
    'rating': 'mean',
    'restaurant_name': 'count'
}).reset_index()
high_perf.columns = ['area', 'avg_rating', 'restaurant_count']
high_perf = high_perf[high_perf['avg_rating'] >= 3.8].sort_values('avg_rating', ascending=False)
print(high_perf)

print("\n✓ Location Analysis Complete!")